# 04 – Pair-laps and target variable

**Goal:** Build the unit of analysis – a pair of consecutive cars in one lap – with the target `overtake` and the features known at the start of the lap; document every exclusion; first overtake rates for the pilot seasons.
**Input:** PostgreSQL tables `laps`, `results`, `races`, `weather` (notebook 03) · **Output:** table `pairs`, `data/processed/pairs.parquet`, `reports/logs/pairs_waterfall.csv`, figures in `reports/figures/`
**Assessment items:** P2 (documented exclusions), P3/B2 (SQL from Python), basis for P4–P7

**Definition.** For every lap *n* ≥ 2, car A on position *p* and car B on position *p*+1 at the end of lap *n*−1 form a pair. `overtake = 1` if B is ahead of A at the end of lap *n*. Only pairs with a gap ≤ 2.0 s at the start of lap *n* are analysed.

**Exclusion rules** (applied in this order, each rule counts what it removes):
1. no lap-*n* data for A or B (retirement, end of race for lapped cars, missing timing)
2. positions not consecutive at lap start
3. pit stop of A or B in lap *n* (positions change in the pit lane, not on track)
4. lap *n* neutralised (safety car, VSC, red flag)
5. restart lap (lap *n*−1 neutralised: cars are bunched up artificially)
6. missing timing / negative gap (inconsistent data)
7. gap > 2.0 s at lap start

**No leakage:** all features use information available at the start of lap *n*: order and gap at the end of lap *n*−1, tyres fitted for lap *n*, mean clean lap time of laps *n*−3 to *n*−1, qualifying, weather at lap start.

In [ ]:
import sys
from pathlib import Path

# Make the repository root importable, whether Jupyter starts in the root or in notebooks/
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))

import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from src.build_pairs import build_pairs, run_from_db
from src.config import ERA_COLORS, GAP_MAX, PILOT_SEASONS
from src.db import get_engine
from src.viz import TEXT_SECONDARY, add_source, save, set_style

set_style()
engine = get_engine()

## 1. Build the pairs (read from and write to PostgreSQL)

In [ ]:
pairs, waterfall = run_from_db(engine)
waterfall

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.6))
steps = waterfall.iloc[1:]
y = np.arange(len(steps))[::-1]
ax.barh(y, steps["removed"], color=ERA_COLORS["E1"], height=0.6)
ax.set_yticks(y, steps["rule"].str.replace(r" \(.*\)", "", regex=True))
ax.grid(axis="x"); ax.grid(axis="y", visible=False)
ax.tick_params(axis="y", length=0)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:,.0f}"))
for yi, v in zip(y, steps["removed"]):
    ax.text(v, yi, f" {v:,}", va="center", fontsize=8, color=TEXT_SECONDARY)
ax.set_xlabel("pair-laps removed")
ax.set_title(f"From {waterfall['remaining'].iloc[0]:,} candidate pair-laps to "
             f"{waterfall['remaining'].iloc[-1]:,} analysed pair-laps")
add_source(fig, f"FastF1 / Jolpica-F1, seasons {min(PILOT_SEASONS)}–{max(PILOT_SEASONS)}, "
                "own calculation")
fig.tight_layout(rect=(0, 0.04, 1, 1))
save(fig, "pilot_pairs_waterfall")
plt.show()

## 2. Target variable: overtake rate per season (SQL)

In [ ]:
by_season = pd.read_sql("""
    SELECT r.season, r.era,
           COUNT(DISTINCT p.race_id) AS races,
           COUNT(*)                  AS pair_laps,
           SUM(p.overtake)           AS overtakes,
           ROUND(100.0 * AVG(p.overtake), 2) AS overtake_rate_pct
    FROM pairs p
    JOIN races r USING (race_id)
    GROUP BY r.season, r.era
    ORDER BY r.season
""", engine)
by_season

The classes are strongly imbalanced (only a few percent of pair-laps end with an overtake). Accuracy is therefore not a suitable main metric for the models later; precision, recall and PR-AUC are.

## 3. Overtake rate by gap at lap start (SQL with `width_bucket`)

In [ ]:
by_gap = pd.read_sql("""
    SELECT r.season, r.era,
           LEAST(width_bucket(p.gap_s, 0, 2.0, 8), 8) AS bucket,
           COUNT(*)        AS pair_laps,
           SUM(p.overtake) AS overtakes
    FROM pairs p
    JOIN races r USING (race_id)
    GROUP BY r.season, r.era, bucket
    ORDER BY r.season, bucket
""", engine)
by_gap["gap_mid_s"] = (by_gap["bucket"] - 0.5) * 0.25
by_gap["rate"] = by_gap["overtakes"] / by_gap["pair_laps"]

# 95 % Wilson confidence interval of a proportion
z = 1.96
n, phat = by_gap["pair_laps"], by_gap["rate"]
centre = (phat + z**2 / (2 * n)) / (1 + z**2 / n)
half = z * np.sqrt(phat * (1 - phat) / n + z**2 / (4 * n**2)) / (1 + z**2 / n)
by_gap["ci_low"], by_gap["ci_high"] = centre - half, centre + half
by_gap

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.2))
for (season, era), g in by_gap.groupby(["season", "era"]):
    color = ERA_COLORS[era]
    ax.fill_between(g["gap_mid_s"], 100 * g["ci_low"], 100 * g["ci_high"],
                    color=color, alpha=0.15, linewidth=0)
    ax.plot(g["gap_mid_s"], 100 * g["rate"], marker="o", color=color,
            label=f"{season} ({era})")
    last = g.iloc[0]
    ax.annotate(f"{season} ({era})", (last["gap_mid_s"], 100 * last["rate"]),
                xytext=(8, 0), textcoords="offset points", color=TEXT_SECONDARY,
                fontsize=9, va="center")
ax.set_xlim(0, GAP_MAX)
ax.set_ylim(bottom=0)
ax.set_xlabel("gap to the car ahead at lap start (s)")
ax.set_ylabel("overtake rate (% of pair-laps)")
ax.set_title("The closer the car behind, the more likely the overtake")
ax.legend(loc="upper right")
add_source(fig, "FastF1 / Jolpica-F1, own calculation; shaded: 95 % Wilson CI")
fig.tight_layout(rect=(0, 0.04, 1, 1))
save(fig, "pilot_overtake_rate_by_gap")
plt.show()

## 4. Sensitivity of the gap threshold

In [ ]:
laps = pd.read_sql("""SELECT race_id, driver_id, lap_number, lap_time_s, time_end_s,
                                position, compound, tyre_life, pit_in, pit_out, track_status
                         FROM laps""", engine)
race_drivers = pd.read_sql("SELECT race_id, driver_id, team, quali_pos FROM results", engine)
races = pd.read_sql("SELECT race_id, season, total_laps FROM races", engine)

rows = []
for gap in (1.0, 1.5, 2.0, 3.0):
    p, _ = build_pairs(laps, race_drivers, races[["race_id", "total_laps"]], gap_max=gap)
    p = p.merge(races[["race_id", "season"]], on="race_id")
    for season, g in p.groupby("season"):
        rows.append({"gap_max_s": gap, "season": season, "pair_laps": len(g),
                     "overtake_rate_pct": round(100 * g["overtake"].mean(), 2)})
pd.DataFrame(rows).pivot(index="gap_max_s", columns="season",
                         values=["pair_laps", "overtake_rate_pct"])

## 5. Why pit-stop laps are excluded (SQL with window functions)

The query below rebuilds the candidate pairs directly in SQL – `LAG()` over the running order of each lap – and compares green-flag pair-laps (gap ≤ 2 s) with and without a pit stop. If a pit stop is involved, the order at the line changes in most cases, because the car in the pit lane loses time, not because it was overtaken on track.

In [ ]:
pd.read_sql("""
    WITH start AS (
        SELECT race_id,
               lap_number + 1                AS lap,
               driver_id                     AS car_b,
               position                      AS pos_b_start,
               time_end_s                    AS t_b,
               LAG(driver_id)  OVER w        AS car_a,
               LAG(position)   OVER w        AS pos_a_start,
               LAG(time_end_s) OVER w        AS t_a
        FROM laps
        WHERE position IS NOT NULL
        WINDOW w AS (PARTITION BY race_id, lap_number ORDER BY position)
    ),
    pair_laps AS (
        SELECT s.t_b - s.t_a                               AS gap_s,
               s.pos_b_start - s.pos_a_start               AS pos_diff,
               lb.position < la.position                   AS order_swapped,
               la.pit_in OR la.pit_out OR lb.pit_in OR lb.pit_out AS pit_stop,
               la.track_status ~ '[4567]' OR lb.track_status ~ '[4567]' AS neutralised
        FROM start s
        JOIN laps la ON la.race_id = s.race_id AND la.driver_id = s.car_a AND la.lap_number = s.lap
        JOIN laps lb ON lb.race_id = s.race_id AND lb.driver_id = s.car_b AND lb.lap_number = s.lap
        WHERE s.car_a IS NOT NULL
    )
    SELECT CASE WHEN pit_stop THEN 'pit stop of A or B in lap n' ELSE 'no pit stop' END AS lap_type,
           COUNT(*)                                       AS pair_laps,
           SUM(order_swapped::int)                        AS order_swaps,
           ROUND(100.0 * AVG(order_swapped::int), 1)      AS swap_rate_pct
    FROM pair_laps
    WHERE NOT neutralised AND gap_s BETWEEN 0 AND 2.0 AND pos_diff = 1
    GROUP BY 1
    ORDER BY 1
""", engine)

## 6. Known limitation: only the order at the line counts

The target uses the order at the end of each lap. Two cases stay invisible:
- **Overtaken and re-overtaken within one lap**, e.g. Bahrain 2021, lap 53 (Verstappen passed Hamilton and had to give the position back) or Bahrain 2022, laps 17–18 (Leclerc and Verstappen swapped twice per lap).
- **On-track overtake in a lap in which one car pits.** Spain 2021, lap 60: Hamilton passed Verstappen for the lead, and Verstappen pitted at the end of that lap, so the pair-lap is removed by rule 3.

In [ ]:
pd.read_sql("""
    SELECT lap_number, driver_id, position, ROUND(time_end_s, 3) AS time_end_s,
           pit_in, pit_out, compound, tyre_life
    FROM laps
    WHERE race_id = 202104 AND lap_number BETWEEN 59 AND 61
      AND driver_id IN ('hamilton', 'max_verstappen')
    ORDER BY lap_number, position
""", engine)